### Ejercicio 2: Experimentar con `printf()` dentro del kernel

In [ ]:
%%writefile excercise2.cu
#include <iostream>

// Kernel function to print thread and block indices
__global__
void printIndices()
{
  printf("Block: %d, Thread: %d\n", blockIdx.x, threadIdx.x);
}

int main(void) {
  // Launch kernel with 2 blocks, each containing 5 threads
  printIndices<<<2, 5>>>();

  // Wait for GPU to finish before accessing on host
  cudaDeviceSynchronize();

  return 0;
}


In [ ]:
!nvcc excercise2.cu -o app_excercise2
!./app_excercise2

**¿Se imprimen en orden secuencial? ¿Por qué o por qué no?**

Al ejecutar el bloque anterior, generalmente notarás que **las impresiones no son necesariamente secuenciales**.

**¿Por qué sucede esto?**
La ejecución en la GPU es masivamente paralela y asíncrona. Los múltiples *thread blocks* se envían a los distintos Streaming Multiprocessors (SMs) y no hay ninguna garantía en el orden en que comenzarán o terminarán. Además, el flujo de salida del `printf` recopila los buffers concurrentemente desde los SMs y los manda al procesador central de forma entrelazada.
Los hilos dentro de un mismo warp (grupo de 32 hilos) pueden aparecer juntos porque en GPU se ejecutan exactamente al mismo tiempo en lockstep, pero entre distintos warps o bloques, el orden es completamente impredecible.